In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from typing import TypedDict, Annotated, NotRequired , Literal
from pydantic import BaseModel, Field 



In [2]:
class SentimentSchema(BaseModel):
      review: str = Field(..., description="The text of the review to analyze")
      sentiment: Literal["positive", "negative"] = Field(..., description="sentiment of the text")

In [16]:
class disagnossSchema(BaseModel):
       issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
       tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
       urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be') 

In [17]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    temperature=0.7,
)

chatModel = ChatHuggingFace(llm=llm)

structured_model = chatModel.with_structured_output(
    SentimentSchema,
    method="json_schema"
)
structured_model2 = chatModel.with_structured_output(
    disagnossSchema,
    method="json_schema"
)

In [19]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive", "negative"]
    diagnosis:dict
    response: str


 


In [20]:
def review_sentiment(state: ReviewState) -> ReviewState:
    review = state["review"]
    output = structured_model.invoke(review)
    
    sentiment = output["sentiment"]  # ✅


    return {"sentiment": sentiment}
    

In [21]:
def check_sentiment(state: ReviewState):
    sentiment = state["sentiment"]

    if sentiment == "positive":
        return "positive_response"
    else:
        return "diagnose_issue"

In [22]:
def positive_response(state: ReviewState) -> ReviewState:
    
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""

    output = chatModel.invoke(prompt).content
    return {"response": output}

In [10]:
def diagnose_issue(state: ReviewState) -> ReviewState:
    
    prompt = f"""The following review is negative. Please analyze the review and provide a diagnosis of the issue mentioned in the review:
    \n\n\"{state['review']}\"\n
    """
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}


In [24]:
def negative_response (state: ReviewState) -> ReviewState:
        diagnosis = state['diagnosis']
        prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
        output = chatModel.invoke(prompt).content
        return {"response": output}

In [ ]:
stateGraph =StateGraph(ReviewState)

stateGraph.add_node("review_sentiment",review_sentiment )
stateGraph.add_node("diagnose_issue", diagnose_issue)
stateGraph.add_node("positive_response", positive_response)
stateGraph.add_node("negative_response", negative_response)

stateGraph.add_edge(START,"review_sentiment")
stateGraph.add_conditional_edges("review_sentiment",check_sentiment)
stateGraph.add_edge("positive_response",END)
stateGraph.add_edge("diagnose_issue","negative_response")
stateGraph.add_edge("negative_response",END)


stateGraph.compile()

HfHubHTTPError: Client error '422 Unprocessable Entity' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: ow7t2DQ-fdNmn-a2ad24a558663725)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422

{'message': 'Input validation error: grammar is not valid: failed to compile grammar', 'type': 'invalid_request_error', 'param': None, 'code': None}
{
  "id": "ow7t2DQ-fdNmn-a2ad24a558663725",
  "error": {
    "message": "Input validation error: grammar is not valid: failed to compile grammar",
    "type": "invalid_request_error",
    "param": null,
    "code": null
  }
}
